# Multi-Label & Imbalance

**Aldo Cortes · October 10–13, 2026**

Deliverable: a working multi-label training run. This notebook explains the implementation and reads the saved weighted BERT smoke-test evidence. It does not rerun training by default.

The original setup used a binary head. Here each chunk has 14 independent clause targets, so several clauses can apply to the same text.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RUN_DIR = ROOT / "results" / "transformer_multilabel" / "weighted_smoke"
config = json.loads((RUN_DIR / "config.json").read_text())
balance = pd.read_csv(RUN_DIR / "class_balance.csv")
log = pd.read_csv(RUN_DIR / "training_log.csv")
metrics = pd.read_csv(RUN_DIR / "validation_metrics.csv")
summary = json.loads((RUN_DIR / "validation_summary.json").read_text())
print(f"Model: {config['model']}")
print(f"Categories: {len(config['categories'])}")
print(f"Completed training steps: {config['training_steps']}")
print(f"Device: {config['device_used']}")
print(f"Run scope: {config['interpretation']}")

Model: bert-base-uncased
Categories: 14
Completed training steps: 5
Device: cpu
Run scope: Pipeline smoke test only


## Independent predictions

A batch has targets and logits shaped `[batch_size, 14]`. Targets are float vectors containing zeros and ones. `BCEWithLogitsLoss` combines sigmoid with binary cross-entropy during training. For inference, sigmoid converts each logit independently to a probability. Softmax would force the categories to compete and would not represent overlapping labels.

The category order below is also saved in the checkpoint.

In [2]:
for index, name in enumerate(config["categories"]):
    print(f"{index:2d}: {name}")
probabilities = np.array([0.90, 0.85, 0.10])
print("\nIllustrative overlapping predictions at threshold 0.5:",
      (probabilities >= 0.5).astype(int).tolist())

 0: Anti-Assignment
 1: Cap on Liability
 2: License Grant
 3: Audit Rights
 4: Post-Termination Services
 5: Termination for Convenience
 6: Exclusivity
 7: Insurance
 8: Minimum Commitment
 9: Revenue/Profit Sharing
10: Non-Transferable License
11: IP Ownership Assignment
12: Non-Compete
13: Change of Control

Illustrative overlapping predictions at threshold 0.5: [1, 1, 0]


## Imbalance handling

For each category, `pos_weight = negative training chunks / positive training chunks`, clipped to 1–50. Categories without positives get neutral weight 1 and an explicit warning. Validation labels never determine these weights.

The cap is a starting choice. A later controlled experiment can compare `--class-weight balanced` with `--class-weight none`. The unweighted loss logged for the same batch is a diagnostic, not an independently trained comparison model.

In [3]:
print(balance.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nAbsent training categories:", config["absent_training_categories"])

                   category  train_positive_chunks  train_negative_chunks  positive_rate  pos_weight
            Anti-Assignment                     18                    354         0.0484     19.6667
           Cap on Liability                      8                    364         0.0215     45.5000
              License Grant                     29                    343         0.0780     11.8276
               Audit Rights                     14                    358         0.0376     25.5714
  Post-Termination Services                     12                    360         0.0323     30.0000
Termination for Convenience                      7                    365         0.0188     50.0000
                Exclusivity                      9                    363         0.0242     40.3333
                  Insurance                      6                    366         0.0161     50.0000
         Minimum Commitment                      7                    365         0.0188   

## Completed weighted training run

The smoke test samples 16 training and 4 validation contracts with seed 42, then runs five optimizer steps. It preserves the saved contract-level splits and excludes the duplicate. Chunk windows reserve two special-token slots within BERT's 512-token maximum, retain 64 tokens of overlap, and use exact tokenizer offsets to assign labels.

In [4]:
print(log.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
assert config["training_steps"] == len(log) and len(log) >= 5
assert np.isfinite(log["loss"]).all()
assert set(config["contract_ids"]["train"]).isdisjoint(config["contract_ids"]["val"])
assert config["test_evaluated"] is False
print("\nTrain/validation chunk counts:", config["chunk_counts"])
print("Elapsed seconds:", config["elapsed_seconds"])
print("Saved model:", RUN_DIR / "model")

 step  epoch   loss  unweighted_bce_same_batch  gradient_norm_before_clip  positive_labels  batch_size
    1      1 2.2036                     0.7192                    18.4573                2           2
    2      1 2.9116                     0.7618                    25.2704                2           2
    3      1 0.7372                     0.7372                     3.0960                0           2
    4      1 1.3633                     0.6967                    18.3836                1           2
    5      1 0.7104                     0.7104                     3.9188                0           2

Train/validation chunk counts: {'train': 372, 'val': 69}
Elapsed seconds: 37.94
Saved model: /Users/aldooo/Documents/BTT/Fall AI Studio/Accenture-1M-contract-review-challenge/results/transformer_multilabel/weighted_smoke/model


## Validation smoke check

These scores cover all 69 chunks from the four selected validation contracts at threshold 0.5. The run checks that sigmoid inference and metric computation work. This small, lightly trained model does not provide evidence of final accuracy or improvement over the baseline. Categories with no positive examples in this validation sample have undefined recall, displayed as zero by the metric implementation. The next milestone will tune thresholds on validation and perform a fuller comparison.

In [5]:
print(json.dumps(summary, indent=2))
print("\nPer-category results:")
print(metrics[["category", "positives", "precision", "recall", "f1"]].to_string(index=False))
print("\nIndependent probabilities for saved validation chunks:")
predictions = pd.read_csv(RUN_DIR / "validation_predictions.csv")
print(predictions.filter(regex=r"^(contract_id|chunk_index|prob__)").head(2).to_string(index=False))

{
  "micro_precision": 0.04285714285714286,
  "micro_recall": 0.4090909090909091,
  "micro_f1": 0.07758620689655173,
  "macro_f1_all_labels": 0.042903848106406746,
  "validation_chunks_evaluated": 69,
  "threshold": 0.5
}

Per-category results:
                   category  positives  precision  recall       f1
            Anti-Assignment          2   0.076923     1.0 0.142857
           Cap on Liability          4   0.000000     0.0 0.000000
              License Grant          8   0.000000     0.0 0.000000
               Audit Rights          5   0.000000     0.0 0.000000
  Post-Termination Services          2   0.029412     1.0 0.057143
Termination for Convenience          0   0.000000     0.0 0.000000
                Exclusivity          4   0.000000     0.0 0.000000
                  Insurance          1   0.000000     0.0 0.000000
         Minimum Commitment          2   0.030769     1.0 0.059701
     Revenue/Profit Sharing          4   0.060606     1.0 0.114286
   Non-Transferabl

## Reproduce or extend

From the repository root with the project environment activated:

```bash
python scripts/transformer_multilabel.py --smoke-test --output-dir results/transformer_multilabel/smoke_repeat
python scripts/transformer_multilabel.py --epochs 3 --batch-size 4 --output-dir results/transformer_multilabel/weighted_full
```

Use `--class-weight none` with the same data, seed and training settings for a controlled unweighted comparison. Existing result directories cannot be overwritten. The saved `model/` is local and ignored by Git because of its size.

Attach the run configuration, training log and class-balance table to the October task. See `MULTILABEL_DELIVERABLE.md` for the full handoff.

References: [PyTorch BCEWithLogitsLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html), [Hugging Face BERT](https://huggingface.co/docs/transformers/model_doc/bert).